# F1 Pit Stop Prediction — Training Framework

Binary classification: predict `PitNextLap` (whether the driver pits on the next lap).

Pipeline: load → EDA → preprocess → CV training → submission.

EDA lives in `eda.ipynb`. This notebook covers feature engineering, cross-validated training, and submission.

In [1]:
# =============================================================
# AUTO-INSTALL + IMPORTS
# =============================================================

import sys
import subprocess

REQUIRED_PACKAGES = [
    "numpy",
    "pandas",
    "matplotlib",
    "seaborn",
    "scikit-learn",
    "lightgbm",
    "xgboost",
    "catboost",
    "scipy"
]

for package in REQUIRED_PACKAGES:
    try:
        __import__(package.replace("-", "_"))
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", package]
        )

# =============================================================
# IMPORTS
# =============================================================

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from sklearn.model_selection import StratifiedGroupKFold

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss
)

import lightgbm as lgb
import xgboost as xgb
import catboost as cb

from scipy.stats import rankdata

# =============================================================
# SETTINGS
# =============================================================

pd.set_option('display.max_columns', 100)

RANDOM_STATE = 42

DATA_DIR = Path('.')

print("All libraries imported successfully.")

/home/propar/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


Installing scikit-learn...
Defaulting to user installation because normal site-packages is not writeable
All libraries imported successfully.


## 1. Load data

In [2]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

TARGET = 'PitNextLap'
ID_COL = 'id'

print('train:', train.shape, '| test:', test.shape)
train.head()

train: (439140, 16) | test: (188165, 15)


,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0
1,1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0


## 2. Data cleaning

Stub — drop/repair invalid rows, handle missing values, fix dtypes. Preserve `(Year, Race, Driver, LapNumber)` ordering if any rows are dropped.

In [ ]:
# =============================================================
# DATA CLEANING (stub)
# =============================================================
# TODO: fill in. Candidates:
#   - drop rows with NaN in critical columns (LapTime, TyreLife, Position)
#   - clip outliers in LapTime_Delta / Cumulative_Degradation
#   - coerce dtypes (Year:int, Stint:int)
#   - drop Pre-Season Testing rows if they hurt CV

def clean(df: pd.DataFrame) -> pd.DataFrame:
    
    df = df.copy()
    return df

train = clean(train)
test  = clean(test)

print('after clean — train:', train.shape, '| test:', test.shape)

## 3. Feature engineering

In [3]:
# =============================================================
# IMPORTS
# =============================================================

import numpy as np
import pandas as pd

import lightgbm as lgb
import xgboost as xgb
import catboost as cb

from scipy.stats import rankdata

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss
)

# =============================================================
# FEATURE ENGINEERING
# =============================================================

def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Improved feature engineering for F1 pit-stop prediction.

    Philosophy:
    - Remove redundant/raw overlapping variables
    - Add interaction-based strategic features
    - Add nonlinear race-window behaviour
    - Preserve primitive variables for tree models
    - Avoid unstable exploding ratios
    """

    df = df.copy()

    # =========================================================
    # BASIC SAFE VARIABLES
    # =========================================================

    eps = 1e-6

    # Remaining race fraction
    df['RemainingRace'] = 1.0 - df['RaceProgress']

    # Mid-race pit-window tendency
    df['PitWindow'] = (
        df['RaceProgress'] *
        (1 - df['RaceProgress'])
    )

    # Late race binary
    df['IsLateRace'] = (
        df['RaceProgress'] > 0.7
    ).astype(int)

    # =========================================================
    # TYRE DYNAMICS
    # =========================================================

    # Pace per tyre age
    df['LapTime_per_TyreLife'] = (
        df['LapTime (s)'] /
        (df['TyreLife'] + eps)
    )

    # Degradation rate per lap
    df['Deg_per_TyreLife'] = (
        df['Cumulative_Degradation'] /
        (df['TyreLife'] + eps)
    )

    # Total tyre stress
    df['TyreStress'] = (
        df['TyreLife'] *
        df['Cumulative_Degradation']
    )

    # Strategic urgency
    df['StrategicUrgency'] = (
        df['TyreLife'] *
        df['Cumulative_Degradation'] *
        df['RemainingRace']
    )

    # Tyre exhaustion factor
    df['TyreExhaustion'] = (
        (df['TyreLife'] ** 2) *
        df['RemainingRace']
    )

    # Tyre cliff detector
    df['TyreCliff'] = (
        df['TyreLife'] *
        df['LapTime_Delta']
    )

    # =========================================================
    # STRATEGY / POSITION FEATURES
    # =========================================================

    # Position pressure
    df['PositionPressure'] = (
        df['Position'] *
        df['RemainingRace']
    )

    # Recovery pressure
    df['RecoveryPressure'] = (
        abs(df['Position_Change']) *
        df['Cumulative_Degradation']
    )

    # =========================================================
    # COMPOUND FEATURES
    # =========================================================

    compound_map = {
        'HARD': 1,
        'MEDIUM': 2,
        'SOFT': 3
    }

    df['CompoundCode'] = (
        df['Compound']
        .astype(str)
        .str.upper()
        .map(compound_map)
        .fillna(2)
    )

    df['CompoundTyreInteraction'] = (
        df['CompoundCode'] *
        df['TyreLife']
    )

    # =========================================================
    # STRATEGIC PIT PRESSURE
    # =========================================================

    df['StrategyPressure'] = (
        df['TyreStress'] *
        df['PitWindow']
    )

    # =========================================================
    # REALISTIC F1 PIT-STOP THINKING
    # =========================================================

    # Can tyre pace loss offset pit-stop cost?
    # Captures whether staying out is becoming expensive
    df['PitOffsetPotential'] = (
        df['LapTime_Delta'] *
        df['RemainingRace'] *
        10
    )

    # Aggressive undercut opportunity
    df['UndercutPotential'] = (
        df['LapTime_Delta'] *
        abs(df['Position_Change']) *
        df['RemainingRace']
    )

    # Race-ending tyre survival pressure
    df['StintSurvivalPressure'] = (
        df['TyreLife'] *
        df['RemainingRace'] *
        df['LapTime_Delta']
    )

    # Pace collapse detector
    df['PaceCollapse'] = (
        df['LapTime_Delta'] *
        df['Cumulative_Degradation']
    )

    # High tyre age late in race
    df['LateRaceTyreRisk'] = (
        df['IsLateRace'] *
        df['TyreLife']
    )

    return df


# =============================================================
# APPLY FEATURE ENGINEERING
# =============================================================

train_fe = add_features(train)
test_fe  = add_features(test)

# =============================================================
# CATEGORICAL FEATURES
# =============================================================

CAT_COLS = [
    'Driver',
    'Compound',
    'Race'
]

for c in CAT_COLS:
    train_fe[c] = train_fe[c].astype('category')
    test_fe[c]  = test_fe[c].astype('category')

# =============================================================
# DROP COLUMNS
# =============================================================

DROP_COLS = [
    ID_COL,
    TARGET,
    'LapNumber',
]

# =============================================================
# FINAL FEATURE LIST
# =============================================================

FEATURES = [
    c for c in train_fe.columns
    if c not in DROP_COLS
]

print(f'{len(FEATURES)} features')
print(FEATURES)

# =============================================================


32 features
['Driver', 'Compound', 'Race', 'Year', 'PitStop', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', 'RemainingRace', 'PitWindow', 'IsLateRace', 'LapTime_per_TyreLife', 'Deg_per_TyreLife', 'TyreStress', 'StrategicUrgency', 'TyreExhaustion', 'TyreCliff', 'PositionPressure', 'RecoveryPressure', 'CompoundCode', 'CompoundTyreInteraction', 'StrategyPressure', 'PitOffsetPotential', 'UndercutPotential', 'StintSurvivalPressure', 'PaceCollapse', 'LateRaceTyreRisk']


## 4. Cross-validation training

Group by `(Race, Year, Driver)` so the same stint doesn't leak across folds.

In [ ]:
# DATA
# =============================================================

# Helper string keys for OOF target encoding groups
def _key(df, cols):
    if len(cols) == 1:
        return df[cols[0]].astype(str)
    return df[cols[0]].astype(str).str.cat(
        [df[c].astype(str) for c in cols[1:]], sep='|'
    )

train_fe['_k_driver']    = _key(train_fe, ['Driver'])
train_fe['_k_race_cmp']  = _key(train_fe, ['Race', 'Compound'])
train_fe['_k_drv_cmp']   = _key(train_fe, ['Driver', 'Compound'])
test_fe['_k_driver']     = _key(test_fe,  ['Driver'])
test_fe['_k_race_cmp']   = _key(test_fe,  ['Race', 'Compound'])
test_fe['_k_drv_cmp']    = _key(test_fe,  ['Driver', 'Compound'])

TE_GROUPS = [
    ('_k_driver',   'TE_Driver'),
    ('_k_race_cmp', 'TE_Race_Compound'),
    ('_k_drv_cmp',  'TE_Driver_Compound'),
]
SMOOTHING = 50.0

def compute_te(train_df, y_tr, val_df, test_df, key_col, smoothing):
    """OOF TE: fit means on train fold only, apply to val and test."""
    global_mean = float(y_tr.mean())
    tmp = pd.DataFrame({'k': train_df[key_col].values, 'y': y_tr.values})
    agg = tmp.groupby('k')['y'].agg(['sum', 'count'])
    smoothed = (agg['sum'] + smoothing * global_mean) / (agg['count'] + smoothing)
    smoothed_dict = smoothed.to_dict()
    tr_te = train_df[key_col].map(smoothed_dict).fillna(global_mean).astype(np.float32).values
    va_te = val_df[key_col].map(smoothed_dict).fillna(global_mean).astype(np.float32).values
    te_te = test_df[key_col].map(smoothed_dict).fillna(global_mean).astype(np.float32).values
    return tr_te, va_te, te_te

y = train_fe[TARGET]

groups = (
    train_fe['Race'].astype(str) + '_' +
    train_fe['Year'].astype(str) + '_' +
    train_fe['Driver'].astype(str)
)

BASE_FEATURES = [
    c for c in train_fe.columns
    if c not in DROP_COLS and not c.startswith('_k_')
]
TE_FEATURES = [name for _, name in TE_GROUPS]
FEATURES = BASE_FEATURES + TE_FEATURES
print(f'{len(BASE_FEATURES)} base + {len(TE_FEATURES)} TE = {len(FEATURES)} features')

# =============================================================
# TUNED PARAMS (T21 carryover): deeper, lower LR, 2-seed
# =============================================================

SEEDS = [42, 2024]

def make_xgb_params(seed):
    return dict(
        n_estimators=6000, learning_rate=0.03, max_depth=7,
        subsample=0.9, colsample_bytree=0.9, min_child_weight=20,
        reg_alpha=0.1, reg_lambda=1.0,
        objective='binary:logistic', eval_metric='auc', seed=seed,
        tree_method='hist', n_jobs=8, early_stopping_rounds=200,
    )

def make_cb_params(seed):
    return dict(
        iterations=6000, learning_rate=0.03, depth=8, l2_leaf_reg=5.0,
        loss_function='Logloss', eval_metric='AUC', random_seed=seed,
        early_stopping_rounds=200, thread_count=8, verbose=0,
    )

def rank_avg(*preds):
    n = len(preds[0])
    return sum(rankdata(p) / n for p in preds) / len(preds)

# =============================================================
# CROSS VALIDATION
# =============================================================

N_SPLITS = 5
cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

oof_xgb = np.zeros(len(train_fe))
oof_cb  = np.zeros(len(train_fe))
test_xgb = np.zeros(len(test_fe))
test_cb  = np.zeros(len(test_fe))

for fold, (tr_idx, va_idx) in enumerate(cv.split(train_fe, y, groups)):

    print(f'\n================ FOLD {fold} ================\n')

    tr_df = train_fe.iloc[tr_idx].copy()
    va_df = train_fe.iloc[va_idx].copy()
    te_df = test_fe.copy()
    y_tr = y.iloc[tr_idx]
    y_va = y.iloc[va_idx]

    # OOF target encoding
    for key_col, te_name in TE_GROUPS:
        tr_te, va_te, te_te = compute_te(tr_df, y_tr, va_df, te_df, key_col, SMOOTHING)
        tr_df[te_name] = tr_te
        va_df[te_name] = va_te
        te_df[te_name] = te_te

    X_tr = tr_df[FEATURES]
    X_va = va_df[FEATURES]
    X_te = te_df[FEATURES]

    X_tr_xgb = X_tr.copy(); X_va_xgb = X_va.copy(); X_te_xgb = X_te.copy()
    for c in CAT_COLS:
        X_tr_xgb[c] = X_tr_xgb[c].cat.codes
        X_va_xgb[c] = X_va_xgb[c].cat.codes
        X_te_xgb[c] = X_te_xgb[c].cat.codes

    cat_idx = [X_tr.columns.get_loc(c) for c in CAT_COLS]

    fold_xgb_va = np.zeros(len(va_idx)); fold_xgb_te = np.zeros(len(test_fe))
    fold_cb_va  = np.zeros(len(va_idx)); fold_cb_te  = np.zeros(len(test_fe))

    for s in SEEDS:
        m = xgb.XGBClassifier(**make_xgb_params(s))
        m.fit(X_tr_xgb, y_tr, eval_set=[(X_va_xgb, y_va)], verbose=False)
        fold_xgb_va += m.predict_proba(X_va_xgb)[:, 1] / len(SEEDS)
        fold_xgb_te += m.predict_proba(X_te_xgb)[:, 1] / len(SEEDS)

        cm = cb.CatBoostClassifier(**make_cb_params(s))
        cm.fit(X_tr, y_tr, eval_set=(X_va, y_va), cat_features=cat_idx, verbose=False)
        fold_cb_va += cm.predict_proba(X_va)[:, 1] / len(SEEDS)
        fold_cb_te += cm.predict_proba(X_te)[:, 1] / len(SEEDS)

    oof_xgb[va_idx] = fold_xgb_va
    oof_cb[va_idx]  = fold_cb_va
    test_xgb += fold_xgb_te / N_SPLITS
    test_cb  += fold_cb_te  / N_SPLITS

    print(f'XGB Fold AUC : {roc_auc_score(y_va, fold_xgb_va):.5f}')
    print(f'CAT Fold AUC : {roc_auc_score(y_va, fold_cb_va):.5f}')

oof_ensemble  = rank_avg(oof_xgb, oof_cb)
test_ensemble = rank_avg(test_xgb, test_cb)

print('\n================ FINAL RESULTS ================\n')
print('ENSEMBLE OOF AUC :', roc_auc_score(y, oof_ensemble))
print('ENSEMBLE OOF AP  :', average_precision_score(y, oof_ensemble))
print('ENSEMBLE OOF LL  :', log_loss(y, oof_ensemble))

submission = pd.DataFrame({ID_COL: test[ID_COL], TARGET: test_ensemble})
submission.to_csv('submission.csv', index=False)
print('\nsubmission.csv saved')


## 5. Feature importance

In [ ]:
imp = feat_importance.mean(axis=1).sort_values(ascending=True)
plt.figure(figsize=(8, max(4, len(imp) * 0.25))); imp.plot.barh(); plt.title('Mean gain importance'); plt.tight_layout(); plt.show()

NameError: name 'feat_importance' is not defined

: 

## 6. Submission

In [ ]:
submission = sample_submission.copy()
submission[TARGET] = test_ensemble
submission.to_csv('submission.csv', index=False)
submission.head()